# Assignment 4 - Part I: Anomaly Detection

MNIST digits 0 - 8 are treated as normal images coming from a healthy asset; digit 9 is the anomaly class. An autoencoder is trained on the normal data only, then any test image it reconstructs badly gets flagged as an anomaly. The threshold on reconstruction error is derived from the test loss statistics, as the spec specifies.

## What this notebook covers for the different grades

- **Grade 3:** load MNIST, normalise, filter digit 9 out of the train pool, build the three data loaders, check loader shapes, plot a sample from each, and explain the anomaly-detection pipeline.
- **Grade 4:** implement a 3-linear-layer-per-side autoencoder, pick MSE + Adam + a sensible learning rate and epoch budget, train, plot the training loss.
- **Grade 5:** compute the test-loss mean and std, set threshold = mean + std, classify every test image, build the confusion matrix.

In [ ]:
import os
import sys
from pathlib import Path

# phm-notebooks/ is one level up from this notebook's folder.
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

from config import *

In [ ]:
# env check
print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device    : {torch.cuda.get_device_name(0)}")
print(f"Conda env : {os.environ.get('CONDA_DEFAULT_ENV', 'unknown')}")

In [ ]:
# Hyperparameters and paths come straight from config.py
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MNIST_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"latent_dim={AE_LATENT_DIM}, epochs={AE_EPOCHS}, lr={AE_LR}, batch={MNIST_BATCH_SIZE}")
print(f"threshold rule  : mean + {AE_THRESHOLD_K} * std on test losses")
print(f"checkpoint path : {MODELS_DIR / AE_CHECKPOINT_NAME}")

## Grade 3 - load and preprocess MNIST

In [ ]:
# ToTensor scales to [0, 1]; Normalize uses the known MNIST stats.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

mnist_train_full = datasets.MNIST(root=str(MNIST_DIR), train=True,  download=True, transform=transform)
mnist_test       = datasets.MNIST(root=str(MNIST_DIR), train=False, download=True, transform=transform)
print(f"full train: {len(mnist_train_full)},  test: {len(mnist_test)}")

In [ ]:
# Keep only digits 0-8 for training. Digit 9 stays only in the test split.
normal_mask = torch.isin(mnist_train_full.targets, torch.tensor(MNIST_NORMAL_DIGITS))
normal_indices = normal_mask.nonzero(as_tuple=True)[0].tolist()
normal_train = Subset(mnist_train_full, normal_indices)

val_size = int(AE_VAL_RATIO * len(normal_train))
train_size = len(normal_train) - val_size
train_split, val_split = random_split(
    normal_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_split, batch_size=MNIST_BATCH_SIZE, shuffle=True,  num_workers=MNIST_NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_split,   batch_size=MNIST_BATCH_SIZE, shuffle=False, num_workers=MNIST_NUM_WORKERS)
test_loader  = DataLoader(mnist_test,  batch_size=MNIST_BATCH_SIZE, shuffle=False, num_workers=MNIST_NUM_WORKERS)

print(f"train {len(train_split)},  val {len(val_split)},  test {len(mnist_test)} (incl. digit 9)")

In [ ]:
# Sanity check: Expected shape should be (batch, 1, 28, 28).
def _peek(loader, name):
    imgs, labels = next(iter(loader))
    print(f"{name:5s}: images {tuple(imgs.shape)}  labels {tuple(labels.shape)}")
    return imgs, labels

train_imgs, train_labels = _peek(train_loader, "train")
test_imgs,  test_labels  = _peek(test_loader,  "test")

fig, ax = plt.subplots(1, 2, figsize=(5, 3))
ax[0].imshow(train_imgs[0].squeeze(), cmap="gray"); ax[0].axis("off")
ax[0].set_title(f"train label {train_labels[0].item()}")
ax[1].imshow(test_imgs[0].squeeze(),  cmap="gray"); ax[1].axis("off")
ax[1].set_title(f"test label {test_labels[0].item()}")
plt.tight_layout(); plt.show()

## Grade 4 - autoencoder model

Three linear layers per side with ReLU between them. No output activation: pixels are already normalised so they go negative, and a sigmoid would clip them. Loss is MSE, optimiser is Adam.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=AE_INPUT_DIM, h1=AE_HIDDEN_1, h2=AE_HIDDEN_2, latent=AE_LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1), 
            nn.ReLU(),
            nn.Linear(h1, h2),        
            nn.ReLU(),
            nn.Linear(h2, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, h2), 
            nn.ReLU(),
            nn.Linear(h2, h1),     
            nn.ReLU(),
            nn.Linear(h1, input_dim),
        )

    def encode(self, x):
        return self.encoder(x.view(x.size(0), -1))

    def decode(self, z):
        return self.decoder(z).view(z.size(0), 1, 28, 28)

    def forward(self, x):
        return self.decode(self.encode(x))

model = Autoencoder().to(device)
print(model)

In [ ]:
# Training loop with simple early stopping. Saves the best checkpoint to models/autoencoder.pt.
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=AE_LR)

train_losses, val_losses = [], []
best_val = float("inf")
patience = 0

for epoch in tqdm(range(AE_EPOCHS), desc="train", unit="epoch"):
    model.train(); 
    total_train_loss = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        loss = criterion(model(imgs), imgs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_train_loss += loss.item()
    train_losses.append(tot / len(train_loader))

    model.eval(); 
    total_val_loss = 0.0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            total_val_loss += criterion(model(imgs), imgs).item()
    val_losses.append(total_val_loss / len(val_loader))

    if val_losses[-1] < best_val - AE_MIN_DELTA:
        best_val = val_losses[-1]; patience = 0
        torch.save(model.state_dict(), MODELS_DIR / AE_CHECKPOINT_NAME)
    else:
        patience += 1
        if patience >= AE_PATIENCE:
            print(f"early stop at epoch {epoch+1}, best val {best_val:.6f}")
            break

print(f"saved {MODELS_DIR / AE_CHECKPOINT_NAME}")

In [ ]:
# Grade 4: training loss curve.
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(train_losses, label="train")
ax.plot(val_losses,   label="val")
ax.set_xlabel("epoch"); 
ax.set_ylabel("MSE"); ax.set_title("Autoencoder training loss")
ax.legend(); 
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Grade 5 - threshold, classify, confusion matrix

Reload the best checkpoint before evaluating so the eval cells are reproducible without retraining. The threshold rule from the spec is `mean + std` of test reconstruction errors.

In [ ]:
# Reload the saved best weights so eval is decoupled from the training run.
model = Autoencoder().to(device)
model.load_state_dict(
    torch.load(MODELS_DIR / AE_CHECKPOINT_NAME, map_location=device, weights_only=True)
)
model.eval()

per_pixel = nn.MSELoss(reduction="none")

error_parts, label_parts, recon_parts, orig_parts = [], [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        recon = model(imgs)
        err   = per_pixel(recon, imgs).mean(dim=[1, 2, 3])
        error_parts.append(err.cpu()); 
        label_parts.append(labels)
        recon_parts.append(recon.cpu()); 
        orig_parts.append(imgs.cpu())

errors  = torch.cat(error_parts).numpy()
labels  = torch.cat(label_parts).numpy()
recons  = torch.cat(recon_parts)
origins = torch.cat(orig_parts)

mu, sd    = float(errors.mean()), float(errors.std())
threshold = mu + AE_THRESHOLD_K * sd
print(f"test errors: mean={mu:.6f}, std={sd:.6f}, threshold={threshold:.6f}")

In [ ]:
# Predict + headline metrics.
y_true = (labels == MNIST_ANOMALY_DIGIT).astype(int)
y_pred = (errors > threshold).astype(int)

acc = float((y_true == y_pred).mean())
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
print(f"accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

## My Interpretation

Sitting the threshold one std above the mean test error makes the detector lean toward sensitivity: recall is high, precision lower. That's the right trade-off for an anomaly detector in a PHM setting - missing a real fault is worse than flagging a borderline normal sample for human review. The histogram below shows the two populations overlap noticeably, which is why no threshold gets perfect separation on raw pixel reconstruction error.

In [ ]:
# For grade-5: Show confusion matrix, error histogram, and a sample-verdicts panel (3 normal + 3 anomaly).

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ax = axes[0]
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["normal", "anomaly"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["normal", "anomaly"])
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

ax = axes[1]
ax.hist(errors[labels != MNIST_ANOMALY_DIGIT], bins=60, alpha=0.6, label="normal (0-8)")
ax.hist(errors[labels == MNIST_ANOMALY_DIGIT], bins=60, alpha=0.6, label="anomaly (9)")
ax.axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
ax.set_xlabel("reconstruction MSE"); ax.set_ylabel("count")
ax.set_title("Reconstruction error by class"); ax.legend()

plt.tight_layout(); 
plt.show()

normal_idx  = np.where(labels != MNIST_ANOMALY_DIGIT)[0][:3]
anomaly_idx = np.where(labels == MNIST_ANOMALY_DIGIT)[0][:3]
picks = list(normal_idx) + list(anomaly_idx)

fig, axes = plt.subplots(6, 3, figsize=(7, 11))
for row, idx in enumerate(picks):
    axes[row, 0].imshow(origins[idx].squeeze(), cmap="gray"); axes[row, 0].axis("off")
    axes[row, 0].set_title(f"orig (label {labels[idx]})", fontsize=9)
    axes[row, 1].imshow(recons[idx].squeeze(),  cmap="gray"); axes[row, 1].axis("off")
    axes[row, 1].set_title("recon", fontsize=9)
    axes[row, 2].axis("off")
    verdict = "ANOMALY" if y_pred[idx] == 1 else "normal"
    color   = "red"     if y_pred[idx] == 1 else "green"
    axes[row, 2].text(0.5, 0.55, verdict, ha="center", va="center",
                      color=color, fontsize=12, weight="bold")
    axes[row, 2].text(0.5, 0.2, f"err={errors[idx]:.4f}", ha="center", va="center", fontsize=8)

plt.suptitle("Sample verdicts: 3 normal + 3 anomaly (orig | recon | verdict)", y=1.0)
plt.tight_layout(); 
plt.show()